# Notebook 3 — Subgroup-Aware RoBERTa Baseline

This notebook tests whether annotator subgroup information helps when it is injected explicitly into the model architecture.

Instead of giving the subgroup as text:

```text
[TARGET_SUBGROUP] women
[TWEET] ...
```

this notebook uses:

```text
tweet → RoBERTa CLS embedding
subgroup_id → learned subgroup embedding

[CLS ; subgroup_embedding] → classifier → distribution
```

This checks whether architectural subgroup conditioning is stronger than a plain text-prefix subgroup token.


## 1. Imports and Configuration

In [5]:
import ast
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

RANDOM_SEED = 42

MODEL_NAME = "roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 192
BATCH_SIZE = 16
EPOCHS = 6
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
SUBGROUP_DIM = 32
DROPOUT = 0.1

EXPERIMENT = "women"  # options: "women", "immigration"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/data")
OUTPUT_DIR = Path("women_subgroup_embedding_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)


Device: cuda


In [6]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## 2. Load Processed Data

In [7]:
women_df = pd.read_parquet(DATA_DIR / "women_processed (1).parquet")
immigration_df = pd.read_parquet(DATA_DIR / "immigration_processed.parquet")

print("Women:", women_df.shape)
print("Immigration:", immigration_df.shape)

display(women_df.head(2))


Women: (3953, 12)
Immigration: (3799, 12)


,comment_id,text,split,num_annotations,overall_counts,overall_distribution,entropy,majority_label,expected_label,subgroup_counts,subgroup_label_counts,subgroup_distributions
0,6,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),train,2,"[2.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0.0,0,0.0,"{'men': 1.0, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1.0}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}"
1,11,"eat my fuck, bitch",validation,2,"[0.0, 1.0, 1.0]","[0.0, 0.5, 0.5]",1.0,1,1.5,"{'men': 1.0, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1.0}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}"


## 3. Expand Comment Rows into Comment–Subgroup Examples

In [8]:
def ensure_dict(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        return ast.literal_eval(value)
    raise TypeError(f"Expected dict or stringified dict, got {type(value)}")


def is_valid_distribution(dist, num_labels: int = NUM_LABELS, tolerance: float = 1e-5) -> bool:
    if dist is None:
        return False
    if len(dist) != num_labels:
        return False
    if any(float(p) < -tolerance for p in dist):
        return False
    return abs(sum(float(p) for p in dist) - 1.0) < tolerance


def expand_subgroup_examples(
    comment_df: pd.DataFrame,
    experiment_name: str,
    allowed_subgroups: list[str] | None = None,
) -> pd.DataFrame:
    rows = []

    for _, row in comment_df.iterrows():
        subgroup_distributions = ensure_dict(row["subgroup_distributions"])
        subgroup_counts = ensure_dict(row["subgroup_counts"])

        for subgroup, target_distribution in subgroup_distributions.items():
            if allowed_subgroups is not None and subgroup not in allowed_subgroups:
                continue

            if not is_valid_distribution(target_distribution):
                continue

            target_distribution = [float(x) for x in target_distribution]

            rows.append({
                "experiment": experiment_name,
                "comment_id": row["comment_id"],
                "split": row["split"],
                "subgroup": subgroup,
                "subgroup_count": int(subgroup_counts.get(subgroup, 0)),
                "text": row["text"],
                "target_distribution": target_distribution,
                "target_majority_label": int(np.argmax(target_distribution)),
                "target_expected_label": float(np.dot(np.arange(NUM_LABELS), target_distribution)),
                "target_entropy": float(entropy(target_distribution, base=2)),
            })

    return pd.DataFrame(rows)


In [9]:
WOMEN_ALLOWED_SUBGROUPS = ["women", "men", "non_binary"]

women_examples = expand_subgroup_examples(
    women_df,
    experiment_name="women",
    allowed_subgroups=WOMEN_ALLOWED_SUBGROUPS,
)

Immigration_ALLOWED_SUBGROUPS = ["neutral","liberal","conservative","slightly_liberal","slightly_conservative","extremely_liberal","extremely_conservative","no_opinion"]
immigration_examples = expand_subgroup_examples(
    immigration_df,
    experiment_name="immigration",
    allowed_subgroups=Immigration_ALLOWED_SUBGROUPS,
)

print("Women examples:", women_examples.shape)
print("Immigration examples:", immigration_examples.shape)

display(women_examples.head())
display(immigration_examples.head())


Women examples: (7962, 10)
Immigration examples: (9090, 10)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0
2,women,11,validation,men,1,"eat my fuck, bitch","[0.0, 0.0, 1.0]",2,2.0,0.0
3,women,11,validation,women,1,"eat my fuck, bitch","[0.0, 1.0, 0.0]",1,1.0,0.0
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,"[0.5, 0.0, 0.5]",0,1.0,1.0


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go st...,"[1.0, 0.0, 0.0]",0,0.0,0.0
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go st...,"[0.0, 0.0, 1.0]",2,2.0,0.0
2,immigration,10,train,liberal,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're ...","[0.0, 0.0, 1.0]",2,2.0,0.0
3,immigration,10,train,neutral,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're ...","[0.0, 0.0, 1.0]",2,2.0,0.0
4,immigration,10,train,slightly_conservative,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're ...","[1.0, 0.0, 0.0]",0,0.0,0.0


## 4. Select Experiment and Create Subgroup IDs

In [10]:
if EXPERIMENT == "women":
    model_df = women_examples.copy()
elif EXPERIMENT == "immigration":
    model_df = immigration_examples.copy()
else:
    raise ValueError("EXPERIMENT must be 'women' or 'immigration'.")

subgroup_to_id = {
    subgroup: idx
    for idx, subgroup in enumerate(sorted(model_df["subgroup"].unique()))
}

id_to_subgroup = {
    idx: subgroup
    for subgroup, idx in subgroup_to_id.items()
}

model_df["subgroup_id"] = model_df["subgroup"].map(subgroup_to_id)

print("Subgroup mapping:")
print(subgroup_to_id)

display(pd.crosstab(model_df["split"], model_df["subgroup"]))
display(model_df.head())


Subgroup mapping:
{'men': 0, 'non_binary': 1, 'women': 2}


subgroup,men,non_binary,women
split,,,
test,576,16,581
train,2723,108,2734
validation,604,16,604


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy,subgroup_id
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0,0
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0,2
2,women,11,validation,men,1,"eat my fuck, bitch","[0.0, 0.0, 1.0]",2,2.0,0.0,0
3,women,11,validation,women,1,"eat my fuck, bitch","[0.0, 1.0, 0.0]",1,1.0,0.0,2
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,"[0.5, 0.0, 0.5]",0,1.0,1.0,0


In [11]:
train_df = model_df[model_df["split"] == "train"].reset_index(drop=True)
val_df = model_df[model_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0


Train: (5565, 11)
Val: (1224, 11)
Test: (1173, 11)


In [12]:
print("Training majority-label distribution:")
display(train_df["target_majority_label"].value_counts(normalize=True).sort_index())

print("Test majority-label distribution:")
display(test_df["target_majority_label"].value_counts(normalize=True).sort_index())


Training majority-label distribution:


target_majority_label
0    0.684277
1    0.071159
2    0.244564
Name: proportion, dtype: float64

Test majority-label distribution:


target_majority_label
0    0.669224
1    0.073316
2    0.257460
Name: proportion, dtype: float64

## 5. Dataset and Dataloader

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [14]:
class SubgroupEmbeddingDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]

        encoding = self.tokenizer(
            row["text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "subgroup_id": torch.tensor(row["subgroup_id"], dtype=torch.long),
            "target_distribution": torch.tensor(row["target_distribution"], dtype=torch.float),
        }


In [15]:
train_dataset = SubgroupEmbeddingDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = SubgroupEmbeddingDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = SubgroupEmbeddingDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}


{'input_ids': torch.Size([16, 192]),
 'attention_mask': torch.Size([16, 192]),
 'subgroup_id': torch.Size([16]),
 'target_distribution': torch.Size([16, 3])}

## 6. Subgroup-Aware RoBERTa Model

In [16]:
class SubgroupAwareRoBERTa(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_subgroups: int,
        subgroup_dim: int = 32,
        num_labels: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.subgroup_embedding = nn.Embedding(
            num_embeddings=num_subgroups,
            embedding_dim=subgroup_dim,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size + subgroup_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, subgroup_id):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        subgroup_embedding = self.subgroup_embedding(subgroup_id)

        combined = torch.cat(
            [cls_embedding, subgroup_embedding],
            dim=1,
        )

        logits = self.classifier(combined)

        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
        }


In [17]:
model = SubgroupAwareRoBERTa(
    model_name=MODEL_NAME,
    num_subgroups=len(subgroup_to_id),
    subgroup_dim=SUBGROUP_DIM,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training steps: 2088
Warmup steps: 208


## 7. Metrics

In [18]:
EPS = 1e-12


def kl_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)


def js_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    values = []
    for true_dist, pred_dist in zip(y_true, y_pred):
        values.append(jensenshannon(true_dist, pred_dist, base=2) ** 2)
    return np.array(values)


def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)


def expected_scores(distributions: np.ndarray) -> np.ndarray:
    labels = np.arange(distributions.shape[1])
    return distributions @ labels


def entropy_values(distributions: np.ndarray) -> np.ndarray:
    return np.array([entropy(dist, base=2) for dist in distributions])


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)

    true_expected = expected_scores(y_true)
    pred_expected = expected_scores(y_pred)

    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    metrics = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "expected_label_mae": float(mean_absolute_error(true_expected, pred_expected)),
    }

    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        metrics["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        metrics["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        metrics["entropy_pearson"] = np.nan
        metrics["entropy_spearman"] = np.nan

    return metrics


## 8. Training Helpers

In [19]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None

    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        subgroup_id = batch["subgroup_id"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                subgroup_id=subgroup_id,
            )

            loss = criterion(outputs["log_probs"], targets)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

        total_loss += loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(avg_loss)

    return metrics, y_true, y_pred


## 9. Train Model

In [20]:
best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / f"{EXPERIMENT}_subgroup_embedding_best_model.pt"

history = []

for epoch in range(1, EPOCHS + 1):

    train_metrics, _, _ = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
    )

    val_metrics, _, _ = run_epoch(
        model,
        val_loader,
        optimizer=None,
        scheduler=None,
    )

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model to {best_model_path}")

history_df = pd.DataFrame(history)
display(history_df)

history_path = OUTPUT_DIR / f"{EXPERIMENT}_subgroup_embedding_training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved:", history_path)


Epoch 1/6
Train KL: 0.7064 | Val KL: 0.6395
Train JS: 0.2827 | Val JS: 0.2362
Val Acc: 0.7214 | Val Macro F1: 0.4852
Saved new best model to women_subgroup_embedding_outputs/women_subgroup_embedding_best_model.pt
Epoch 2/6
Train KL: 0.5982 | Val KL: 0.6325
Train JS: 0.2309 | Val JS: 0.2404
Val Acc: 0.7083 | Val Macro F1: 0.4683
Saved new best model to women_subgroup_embedding_outputs/women_subgroup_embedding_best_model.pt
Epoch 3/6
Train KL: 0.5591 | Val KL: 0.6285
Train JS: 0.2149 | Val JS: 0.2320
Val Acc: 0.7279 | Val Macro F1: 0.4847
Saved new best model to women_subgroup_embedding_outputs/women_subgroup_embedding_best_model.pt
Epoch 4/6
Train KL: 0.5180 | Val KL: 0.6610
Train JS: 0.2004 | Val JS: 0.2312
Val Acc: 0.7181 | Val Macro F1: 0.4763
Epoch 5/6
Train KL: 0.4809 | Val KL: 0.7002
Train JS: 0.1871 | Val JS: 0.2292
Val Acc: 0.7181 | Val Macro F1: 0.4774
Epoch 6/6
Train KL: 0.4522 | Val KL: 0.6997
Train JS: 0.1760 | Val JS: 0.2317
Val Acc: 0.7157 | Val Macro F1: 0.4717


,epoch,train_kl_mean,train_js_mean,train_cross_entropy_mean,train_accuracy,train_macro_f1,train_expected_label_mae,train_entropy_pearson,train_entropy_spearman,train_loss,val_kl_mean,val_js_mean,val_cross_entropy_mean,val_accuracy,val_macro_f1,val_expected_label_mae,val_entropy_pearson,val_entropy_spearman,val_loss
0,1,0.706397,0.282739,0.787003,0.705481,0.416926,0.645655,0.092739,0.078206,0.706397,0.639501,0.236212,0.712631,0.721405,0.485205,0.523977,0.124608,0.122264,0.639501
1,2,0.598232,0.230943,0.678838,0.731357,0.476567,0.528641,0.145370,0.131792,0.598232,0.632505,0.240434,0.705635,0.708333,0.468278,0.528694,0.125145,0.108459,0.632505
2,3,0.559066,0.214915,0.639672,0.751123,0.493712,0.481782,0.174938,0.165466,0.559066,0.628467,0.232012,0.701596,0.727941,0.484705,0.515323,0.134183,0.117485,0.628467
3,4,0.518029,0.200411,0.598636,0.759748,0.500488,0.445801,0.204621,0.198098,0.518029,0.661009,0.231156,0.734139,0.718137,0.476255,0.501259,0.124708,0.111657,0.661009
4,5,0.480921,0.187104,0.561527,0.776999,0.513190,0.413095,0.212332,0.200562,0.480921,0.700155,0.229200,0.773285,0.718137,0.477370,0.494378,0.112282,0.106354,0.700155
5,6,0.452150,0.176000,0.532756,0.787242,0.522461,0.387063,0.236091,0.226204,0.452150,0.699703,0.231678,0.772833,0.715686,0.471712,0.496149,0.112876,0.107832,0.699703


Saved: women_subgroup_embedding_outputs/women_subgroup_embedding_training_history.csv


## 10. Test Evaluation

In [21]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics, y_true_test, y_pred_test = run_epoch(
    model,
    test_loader,
    optimizer=None,
    scheduler=None,
)

display(test_metrics)

metrics_path = OUTPUT_DIR / f"{EXPERIMENT}_subgroup_embedding_test_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved:", metrics_path)


/tmp/ipykernel_20760/331801052.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.6606642007827759,
 'js_mean': 0.24147758986736392,
 'cross_entropy_mean': 0.7441272139549255,
 'accuracy': 0.7058823529411765,
 'macro_f1': 0.4601169468829949,
 'expected_label_mae': 0.5511709583342831,
 'entropy_pearson': 0.13688025331855752,
 'entropy_spearman': 0.1295693493798995,
 'loss': 0.6606641483809942}

Saved: women_subgroup_embedding_outputs/women_subgroup_embedding_test_metrics.json


## 11. Save Predictions

In [22]:
predictions_df = test_df.copy()

predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)

predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)

predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

display(predictions_df.head())

predictions_path = OUTPUT_DIR / f"{EXPERIMENT}_subgroup_embedding_test_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)

print("Saved:", predictions_path)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy,subgroup_id,true_distribution,pred_distribution,pred_majority_label,pred_expected_label,pred_entropy,kl,js,cross_entropy
0,women,63,test,men,1,Demand she pay and topple the patriarchy!,"[1.0, 0.0, 0.0]",0,0.0,0.0,0,"[1.0, 0.0, 0.0]","[0.97150886, 0.0122038545, 0.016287237]",0,0.044778,0.214835,0.028905,0.014394,0.028905
1,women,63,test,women,2,Demand she pay and topple the patriarchy!,"[1.0, 0.0, 0.0]",0,0.0,0.0,2,"[1.0, 0.0, 0.0]","[0.96404314, 0.014582461, 0.021374412]",0,0.057331,0.258463,0.036619,0.018216,0.036619
2,women,193,test,men,1,I will personally rape your mother,"[0.0, 0.0, 1.0]",2,2.0,0.0,0,"[0.0, 0.0, 1.0]","[0.8859286, 0.044611502, 0.069459856]",0,0.183531,0.622211,2.667006,0.814565,2.667006
3,women,193,test,women,1,I will personally rape your mother,"[0.0, 0.0, 1.0]",2,2.0,0.0,2,"[0.0, 0.0, 1.0]","[0.8551925, 0.052286513, 0.09252102]",0,0.237329,0.733329,2.380319,0.771402,2.380319
4,women,209,test,men,1,I hope she's been sterilized,"[1.0, 0.0, 0.0]",0,0.0,0.0,0,"[1.0, 0.0, 0.0]","[0.9493736, 0.022140754, 0.028485715]",0,0.079112,0.339103,0.051953,0.025788,0.051953


Saved: women_subgroup_embedding_outputs/women_subgroup_embedding_test_predictions.parquet


## 12. Diagnostic Evaluation

In [23]:
true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())


Confusion matrix:
[[625   0 160]
 [ 48   0  38]
 [ 99   0 203]]

Classification report:
              precision    recall  f1-score   support

           0       0.81      0.80      0.80       785
           1       0.00      0.00      0.00        86
           2       0.51      0.67      0.58       302

    accuracy                           0.71      1173
   macro avg       0.44      0.49      0.46      1173
weighted avg       0.67      0.71      0.69      1173


Predicted label distribution:


0    0.658142
2    0.341858
Name: proportion, dtype: float64


True label distribution:


0    0.669224
1    0.073316
2    0.257460
Name: proportion, dtype: float64

In [24]:
print("Mean KL by true majority label:")
display(
    predictions_df
    .groupby("target_majority_label")
    .agg(
        n=("comment_id", "count"),
        mean_kl=("kl", "mean"),
        mean_js=("js", "mean"),
        mean_target_entropy=("target_entropy", "mean"),
        mean_pred_entropy=("pred_entropy", "mean"),
    )
)

print("Average predicted distribution by true majority label:")
for label in [0, 1, 2]:
    subset = predictions_df[predictions_df["target_majority_label"] == label]
    avg_pred = np.vstack(subset["pred_distribution"].to_numpy()).mean(axis=0)
    print(label, avg_pred)


Mean KL by true majority label:


,n,mean_kl,mean_js,mean_target_entropy,mean_pred_entropy
target_majority_label,,,,,
0,785,0.354950,0.146096,0.142549,0.757125
1,86,2.445053,0.733496,0.197674,1.004678
2,302,0.947182,0.349294,0.040868,1.123785


Average predicted distribution by true majority label:
0 [0.7426834  0.05708002 0.20023647]
1 [0.57548827 0.07258932 0.35192245]
2 [0.4345873  0.07648104 0.48893192]


In [25]:
print("Performance by subgroup:")

subgroup_rows = []

for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())

    row = {
        "subgroup": subgroup,
        "n": len(group),
        **compute_metrics(y_true, y_pred),
    }

    subgroup_rows.append(row)

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")
display(subgroup_metrics_df)

subgroup_metrics_path = OUTPUT_DIR / f"{EXPERIMENT}_subgroup_embedding_subgroup_metrics.csv"
subgroup_metrics_df.to_csv(subgroup_metrics_path, index=False)
print("Saved:", subgroup_metrics_path)


Performance by subgroup:


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,non_binary,16,0.481774,0.202109,0.481774,0.812500,0.539855,0.476763,NaN,NaN
0,men,576,0.647999,0.235380,0.740593,0.696181,0.440671,0.541361,0.203743,0.195788
2,women,581,0.678147,0.248607,0.754856,0.712565,0.474826,0.562945,0.065834,0.055050


Saved: women_subgroup_embedding_outputs/women_subgroup_embedding_subgroup_metrics.csv


## 13. Interpretation Guide

Compare this notebook against the old subgroup-token baseline.

Main question:

```text
Does explicit subgroup embedding improve over subgroup-as-text?
```

Look especially at:

- overall KL / JS
- predicted label distribution
- label-1 recall
- label-2 recall
- mean KL by true majority label
- performance by subgroup

If this model improves, then annotator identity is useful but needed stronger architectural conditioning.
If it does not improve, then subgroup identity may not be a strong signal in this dataset.


In [26]:
@torch.no_grad()
def predict_distribution(text, subgroup):

    model.eval()

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    subgroup_id = torch.tensor(
        [subgroup_to_id[subgroup]],
        dtype=torch.long
    ).to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        subgroup_id=subgroup_id
    )

    probs = outputs["probs"].cpu().numpy()[0]

    return probs

In [27]:
from scipy.spatial.distance import jensenshannon
import itertools

@torch.no_grad()
def predict_distribution(text, subgroup):
    model.eval()

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    subgroup_id = torch.tensor(
        [subgroup_to_id[subgroup]],
        dtype=torch.long
    ).to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        subgroup_id=subgroup_id
    )

    return outputs["probs"].cpu().numpy()[0]

In [28]:
ideology_groups = list(subgroup_to_id.keys())
print("Ideology groups:", ideology_groups)
counterfactual_rows = []

for _, row in test_df.drop_duplicates("comment_id").iterrows():

    text = row["text"]

    preds_by_group = {
        group: predict_distribution(text, group)
        for group in ideology_groups
    }

    pairwise_js = []

    for group_a, group_b in itertools.combinations(ideology_groups, 2):

        js = jensenshannon(
            preds_by_group[group_a],
            preds_by_group[group_b],
            base=2
        ) ** 2

        pairwise_js.append(js)

    counterfactual_rows.append({
        "comment_id": row["comment_id"],
        "text": text,
        "mean_pairwise_js": float(np.mean(pairwise_js)),
        "max_pairwise_js": float(np.max(pairwise_js)),
    })

cf_ideology_df = pd.DataFrame(counterfactual_rows)

print("Mean ideology counterfactual JS:", cf_ideology_df["mean_pairwise_js"].mean())
print("Median ideology counterfactual JS:", cf_ideology_df["mean_pairwise_js"].median())
print("Mean max-pairwise JS:", cf_ideology_df["max_pairwise_js"].mean())
print("Median max-pairwise JS:", cf_ideology_df["max_pairwise_js"].median())

display(
    cf_ideology_df
    .sort_values("max_pairwise_js", ascending=False)
    .head(30)
)

Ideology groups: ['men', 'non_binary', 'women']
Mean ideology counterfactual JS: 0.0011447660689600054
Median ideology counterfactual JS: 0.0012613102364479032
Mean max-pairwise JS: 0.0021213933572777504
Median max-pairwise JS: 0.002258994470408296


,comment_id,text,mean_pairwise_js,max_pairwise_js
410,32158,Bitch I seen you last night lol you was lit kissing your man and shit lmao okkuurrrr,0.002485,0.004960
80,4904,She looks like the kind of retard that chews the Aspirin.,0.002445,0.004684
262,19671,Deserves it stupid bitch,0.002335,0.004654
366,27818,One of these chicks need to be arrested. Register em as a sex offender. Kill all that shit URL,0.002317,0.004582
212,14627,Shut that cunts mouth before I come over there and fuck-start her head!,0.002285,0.004554
279,20753,Bitches who stay with abusive men are so stupid like damn bitch I wanna punch you too,0.002263,0.004510
397,31287,you know whats annoying? feminists who feel its needed to go to the extent of creating massive double standards and screwing over men then calling the men sexist cunts for trying to defend themsel...,0.002238,0.004451
111,7288,Glad to see bitch still has a use. Submissiveness doesn't die with age.,0.002240,0.004435
540,44397,That's not accurate at all. He slut shamed her,0.002268,0.004435
491,39119,A girl KNOWS when a bitch likes her boyfriend,0.002208,0.004388


In [29]:
def pairwise_counterfactual_js(group_a, group_b):
    rows = []

    for _, row in test_df.drop_duplicates("comment_id").iterrows():
        text = row["text"]

        pred_a = predict_distribution(text, group_a)
        pred_b = predict_distribution(text, group_b)

        js = jensenshannon(
            pred_a,
            pred_b,
            base=2
        ) ** 2

        rows.append({
            "comment_id": row["comment_id"],
            "text": text,
            "group_a": group_a,
            "group_b": group_b,
            "js": float(js),
            f"pred_{group_a}": pred_a,
            f"pred_{group_b}": pred_b,
        })

    return pd.DataFrame(rows)


extreme_df = pairwise_counterfactual_js(
    "extremely_liberal",
    "extremely_conservative"
)

print("Extreme liberal vs extreme conservative mean JS:", extreme_df["js"].mean())
print("Extreme liberal vs extreme conservative median JS:", extreme_df["js"].median())

display(
    extreme_df
    .sort_values("js", ascending=False)
    .head(30)
)

KeyError: 'extremely_liberal'

In [ ]:
from scipy.spatial.distance import jensenshannon
import numpy as np
import pandas as pd
import ast

def valid_dist(x):
    if x is None:
        return False
    try:
        arr = np.array(x, dtype=float)
    except Exception:
        return False
    if arr.shape[0] != 3:
        return False
    if np.isnan(arr).any():
        return False
    if arr.sum() <= 0:
        return False
    return True


rows = []

for _, row in immigration_df.iterrows():

    subgroup_dists = row["subgroup_distributions"]

    if isinstance(subgroup_dists, str):
        subgroup_dists = ast.literal_eval(subgroup_dists)

    dist_a = subgroup_dists.get("extremely_liberal")
    dist_b = subgroup_dists.get("extremely_conservative")

    if not valid_dist(dist_a) or not valid_dist(dist_b):
        continue

    dist_a = np.array(dist_a, dtype=float)
    dist_b = np.array(dist_b, dtype=float)

    js = jensenshannon(dist_a, dist_b, base=2) ** 2

    rows.append({
        "comment_id": row["comment_id"],
        "text": row["text"],
        "js": float(js),
        "EL_dist": dist_a,
        "EC_dist": dist_b,
    })

true_el_ec_df = pd.DataFrame(rows)

print("N overlapping EL/EC comments:", len(true_el_ec_df))
print("Mean true JS:", true_el_ec_df["js"].mean())
print("Median true JS:", true_el_ec_df["js"].median())

display(
    true_el_ec_df
    .sort_values("js", ascending=False)
    .head(30)
)

N overlapping EL/EC comments: 101
Mean true JS: 0.22725512987854235
Median true JS: 0.0


,comment_id,text,js,EL_dist,EC_dist
100,49332,Send all these asses back !,1.000000,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]"
84,43425,Das what happen when USA is bombing other countries and Killing innosent peoples .. now try to stop this one. Kongo for life and Afrika for life..,1.000000,"[1.0, 0.0, 0.0]","[0.0, 0.0, 1.0]"
59,30714,Your prime minister is a corrupt racist scumbag who hates Arabs and Palestinians though. You should go back to Israel and fix it so that my tax dollars can stop being wasted on bullets used to mur...,1.000000,"[1.0, 0.0, 0.0]","[0.0, 0.0, 1.0]"
64,32928,Thread reeks of jealous pakis and racist brits. Massive DUB URL,1.000000,"[0.0, 1.0, 0.0]","[1.0, 0.0, 0.0]"
69,39456,"Motherfuckers stupid retards Abwehr degenerate German European Israeli ""Jews"" which have nothing to do with the real Hebrews decided to hijack the Western history and CIVILIZATION! - Google Search...",1.000000,"[0.0, 1.0, 0.0]","[0.0, 0.0, 1.0]"
72,40275,I wish the Natives had the same energy and hate these evil folks that's metally broken in Amerikkka have and had when chris brought his evil arse to Amerikkka. We wouldn't be having this conversat...,1.000000,"[0.5, 0.0, 0.5]","[0.0, 1.0, 0.0]"
74,40531,American scum is already attacking Iran.,1.000000,"[1.0, 0.0, 0.0]","[0.0, 0.0, 1.0]"
14,18346,">I am sorry did i hurt your white pride ? In what world does India have more rape per capita than USA ? I guess some people judge humans based on skin color. I'm Pakistani, and I also believe ther...",1.000000,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]"
13,18268,You are really fucking stupid you ignorant low iq american... have you been outside your country? You sound so fucking stupid they should put you down cunt,1.000000,"[1.0, 0.0, 0.0]","[0.0, 0.0, 1.0]"
81,42851,Deport all wops back to the old country.,1.000000,"[0.0, 0.0, 1.0]","[1.0, 0.0, 0.0]"


In [ ]:
def majority_label(dist):
    return int(np.argmax(np.array(dist, dtype=float)))


true_el_ec_df["EL_label"] = true_el_ec_df["EL_dist"].apply(majority_label)
true_el_ec_df["EC_label"] = true_el_ec_df["EC_dist"].apply(majority_label)

true_el_ec_df["label_pair"] = (
    true_el_ec_df["EL_label"].astype(str)
    + "-"
    + true_el_ec_df["EC_label"].astype(str)
)

display(
    true_el_ec_df
    .groupby("label_pair")
    .agg(
        n=("comment_id", "count"),
        mean_js=("js", "mean"),
        median_js=("js", "median"),
    )
    .sort_values("n", ascending=False)
)

,n,mean_js,median_js
label_pair,,,
0-0,51,0.023588,0.0
2-2,28,0.026777,0.0
0-2,7,1.000000,1.0
0-1,5,1.000000,1.0
2-0,4,1.000000,1.0
1-2,2,1.000000,1.0
2-1,2,1.000000,1.0
1-0,1,1.000000,1.0
1-1,1,0.000000,0.0


In [ ]:
display(
    true_el_ec_df
    .query("js > 0")
    .groupby("label_pair")
    .agg(
        n=("comment_id", "count"),
        mean_js=("js", "mean"),
        median_js=("js", "median"),
    )
    .sort_values("n", ascending=False)
)

,n,mean_js,median_js
label_pair,,,
2-2,12,0.062480,0.025329
0-0,7,0.171858,0.155639
0-2,7,1.000000,1.000000
0-1,5,1.000000,1.000000
2-0,4,1.000000,1.000000
1-2,2,1.000000,1.000000
2-1,2,1.000000,1.000000
1-0,1,1.000000,1.000000


In [ ]:
true_el_ec_df["involves_label_1"] = (
    (true_el_ec_df["EL_label"] == 1)
    | (true_el_ec_df["EC_label"] == 1)
)

display(
    true_el_ec_df
    .groupby("involves_label_1")
    .agg(
        n=("comment_id", "count"),
        mean_js=("js", "mean"),
        median_js=("js", "median"),
    )
)

,n,mean_js,median_js
involves_label_1,,,
False,90,0.143920,0.0
True,11,0.909091,1.0


In [ ]:
display(
    true_el_ec_df
    .query("js > 0")
    .groupby("involves_label_1")
    .agg(
        n=("comment_id", "count"),
        mean_js=("js", "mean"),
        median_js=("js", "median"),
    )
)

,n,mean_js,median_js
involves_label_1,,,
False,30,0.431759,0.179214
True,10,1.000000,1.000000


In [ ]:
comparison_df = true_el_ec_df.merge(
    extreme_df[[
        "comment_id",
        "js",
        "pred_extremely_liberal",
        "pred_extremely_conservative"
    ]],
    on="comment_id",
    suffixes=("_true", "_model")
)

comparison_df = comparison_df.rename(columns={
    "js_true": "true_js",
    "js_model": "model_js"
})

comparison_df["capture_ratio"] = (
    comparison_df["model_js"] /
    (comparison_df["true_js"] + 1e-12)
)

display(comparison_df.head())

,comment_id,text,true_js,EL_dist,EC_dist,EL_label,EC_label,label_pair,involves_label_1,model_js,pred_extremely_liberal,pred_extremely_conservative,capture_ratio
0,13695,Yes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithole as ...,0.311278,"[0.5, 0.0, 0.5]","[1.0, 0.0, 0.0]",0,0,0-0,False,0.000342,"[0.6819129, 0.11475086, 0.20333622]","[0.69854206, 0.102396846, 0.19906108]",1.099420e-03
1,20016,Keep the fucking Mexican scum rapists and murderers out of the country and the fucking Arab Islamic terrorists out of the U.S. too!! No Syrian refugees either!! Build a wall and keep all of these ...,0.004567,"[0.08421052992343903, 0.10526315867900848, 0.8105263113975525]","[0.06666667014360428, 0.06666667014360428, 0.8666666746139526]",2,2,2-2,False,0.000227,"[0.20564345, 0.096090235, 0.69826627]","[0.2027107, 0.08656365, 0.71072567]",4.961086e-02
2,20033,"STOP SUCKING THAT FATASS' DICK, YOU NIGGERLOVING KIKE",0.000000,"[0.0, 0.0, 1.0]","[0.0, 0.0, 1.0]",2,2,2-2,False,0.000279,"[0.14841586, 0.080010474, 0.7715737]","[0.14178166, 0.07151637, 0.7867019]",2.794933e+08
3,20062,I've always been proud of the Igbo tribe and our culture. Yesterday made me even more proud. You people are too fine abeg. OgbakoUmuIgboDay,0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0-0,False,0.000351,"[0.9419692, 0.034675945, 0.023354875]","[0.95169693, 0.028031966, 0.020271135]",3.510777e+08
4,37246,racists b like go back to ur country like ok stop bombing it n maybe i will,0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0-0,False,0.000381,"[0.58269054, 0.13809553, 0.27921396]","[0.6010187, 0.1241873, 0.27479404]",3.812121e+08


In [ ]:
print("Mean true JS:", comparison_df["true_js"].mean())
print("Mean model JS:", comparison_df["model_js"].mean())
print("Mean capture ratio:", comparison_df["capture_ratio"].mean())
print("Median capture ratio:", comparison_df["capture_ratio"].median())

Mean true JS: 0.16448066401642414
Mean model JS: 0.0003310633278506607
Mean capture ratio: 224576822.36447835
Median capture ratio: 211134988.39196873


In [ ]:
display(
    comparison_df
    .groupby("label_pair")
    .agg(
        n=("comment_id", "count"),
        mean_true_js=("true_js", "mean"),
        mean_model_js=("model_js", "mean"),
        mean_capture_ratio=("capture_ratio", "mean"),
        median_capture_ratio=("capture_ratio", "median"),
    )
    .sort_values("mean_true_js", ascending=False)
)

,n,mean_true_js,mean_model_js,mean_capture_ratio,median_capture_ratio
label_pair,,,,,
0-1,1,1.000000,0.000283,2.830846e-04,2.830846e-04
0-0,4,0.077820,0.000429,3.435862e+08,3.661449e+08
2-2,3,0.001522,0.000216,1.407567e+08,1.427767e+08


In [ ]:
comparison_nonzero = comparison_df[
    comparison_df["true_js"] > 0
]

comparison_nonzero["capture_ratio"] = (
    comparison_nonzero["model_js"]
    /
    comparison_nonzero["true_js"]
)

print(
    "Mean capture ratio:",
    comparison_nonzero["capture_ratio"].mean()
)

print(
    "Median capture ratio:",
    comparison_nonzero["capture_ratio"].median()
)

Mean capture ratio: 0.01699778781665204
Median capture ratio: 0.001099419760366172


/tmp/ipykernel_2605/3760677297.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comparison_nonzero["capture_ratio"] = (
